# Heimdall Anomaly Detection - LoRA Training
Model: Qwen/Qwen2.5-0.5B-Instruct
Dataset: Kaggle logging-and-monitoring-anomalies converted to JSONL

In [ ]:
# Install dependencies
%pip install -q unsloth transformers datasets trl peft accelerate bitsandbytes

In [ ]:
# Import libraries
from datasets import load_dataset
from transformers import TrainingArguments
import json

In [ ]:
# Load dataset (upload kaggle_train.jsonl and kaggle_test.jsonl to Colab first)
dataset = load_dataset('json', data_files={'train': 'kaggle_train.jsonl', 'test': 'kaggle_test.jsonl'})
print(f"Train samples: {len(dataset['train'])}")
print(f"Test samples: {len(dataset['test'])}")

In [ ]:
# Load model with 4-bit quantization
from unsloth import FastLanguageModel
import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
max_seq_length = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = torch.float16,
    load_in_4bit = True,
)

In [ ]:
# LoRA configuration
lora_config = {
    'r': 64,
    'lora_alpha': 128,
    'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    'lora_dropout': 0.05,
}

model = FastLanguageModel.get_peft_model(model, lora_config)
print('LoRA applied')

In [ ]:
# Training
from trl import SFTTrainer

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset['train'],
    eval_dataset = dataset['test'],
    dataset_text_field = 'text',  # Adjust based on actual field
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 4,
        per_device_eval_batch_size = 4,
        gradient_accumulation_steps = 2,
        warmup_steps = 20,
        max_steps = 200,
        learning_rate = 2e-5,
        fp16 = True,
        logging_steps = 20,
        output_dir = 'outputs/heimdall-0.5b-kaggle',
    ),
)

trainer.train()